# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The same lane and slice as in w03_data_contact.ipynb: fact_contact_daily_performance, month=2026-03, know half = days 1 to 15, outcome half = days 16 to 31. This time I won’t “floor” the rows with lower demand; I will leave them as is and address missingness

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, numpy as np, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Known half (days 1-15) — no demand floor this time, so real missingness shows up.
known = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                          AS impressions,
        SUM(gsc_clicks)                                               AS clicks,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END)  AS avg_position,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)            AS ga4_days_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions END) AS ga4_sessions
    FROM {MARCH}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
""").df()

# Outcome half (days 16-31) — used ONLY to build the label, never as a feature.
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS outcome_impressions
    FROM {MARCH}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY 1, 2
""").df()

data = known.merge(outcome, on=["client_hash_id", "content_hash_id"], how="left")
data["outcome_impressions"] = data["outcome_impressions"].fillna(0)
print(f"{len(data):,} content items, known half of March 2026")


Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

319,759 content items, known half of March 2026


In [2]:
# --- Engineer features + handle missing/categorical explicitly ---
df = data.copy()

# Numeric: fill count-like columns with 0 (a genuine zero — no rows means no activity that half)
for col in ["impressions", "clicks", "days_with_impressions", "ga4_days_available", "ga4_sessions"]:
    df[col] = df[col].fillna(0)

# avg_position is different: NaN here means "never had an impression" -- NOT "ranked #0".
# Filling with 0 would fabricate a top rank. Instead: flag it, then fill with a sentinel
# worse than any real rank, so a tree can split it away from real positions.
df["no_position_data"] = df["avg_position"].isna().astype(int)
df["avg_position_filled"] = df["avg_position"].fillna(999)

# Engineered numeric feature: log1p smooths the long right tail of impressions
df["log_impressions"] = np.log1p(df["impressions"])

# Engineered ratio feature: guard divide-by-zero explicitly rather than letting it silently NaN
df["ctr_pct"] = np.where(df["impressions"] > 0, df["clicks"] / df["impressions"] * 100, 0.0)

# Categorical: bucket position into tiers, one-hot encode (includes its own "no_data" bucket
# so the bucket and the numeric fill agree with each other instead of contradicting).
def bucket(row):
    if row["no_position_data"]:
        return "no_data"
    p = row["avg_position"]
    if p <= 10: return "top_10"
    if p <= 20: return "11_20"
    if p <= 50: return "21_50"
    return "50_plus"

df["position_bucket"] = df.apply(bucket, axis=1)
bucket_dummies = pd.get_dummies(df["position_bucket"], prefix="pos")
df = pd.concat([df, bucket_dummies], axis=1)

# The label — built ONLY from the outcome half, never touched by anything above.
df["declined"] = (df["outcome_impressions"] < 0.8 * df["impressions"].clip(lower=1)).astype(int)

FEATURES = ["log_impressions", "clicks", "avg_position_filled", "no_position_data",
            "days_with_impressions", "ctr_pct", "ga4_days_available"] + list(bucket_dummies.columns)
print("Feature columns:", FEATURES)
df[FEATURES + ["declined"]].head()


Feature columns: ['log_impressions', 'clicks', 'avg_position_filled', 'no_position_data', 'days_with_impressions', 'ctr_pct', 'ga4_days_available', 'pos_11_20', 'pos_21_50', 'pos_50_plus', 'pos_no_data', 'pos_top_10']


,log_impressions,clicks,avg_position_filled,no_position_data,days_with_impressions,ctr_pct,ga4_days_available,pos_11_20,pos_21_50,pos_50_plus,pos_no_data,pos_top_10,declined
0,4.718499,0.0,5.222776,0,13,0.000000,0,False,False,False,False,True,1
1,3.663562,1.0,4.638889,0,9,2.631579,0,False,False,False,False,True,1
2,5.393628,1.0,3.737399,0,15,0.456621,0,False,False,False,False,True,0
3,3.044522,0.0,3.597222,0,9,0.000000,0,False,False,False,False,True,1
4,7.309881,0.0,6.156643,0,14,0.000000,0,False,False,False,False,True,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


| Feature | Meaning | Missing handling | Categorical? | Available when? |
|---|---|---|---|---|
| `log_impressions` | log1p of days-1–15 impressions | count columns fill with 0 (a real zero) | no | end of day 15 |
| `clicks` | sum of days-1–15 clicks | fill 0 | no | end of day 15 |
| `avg_position_filled` | mean rank on days with impressions, days 1–15 | `NaN` (never ranked) → sentinel `999`, paired with the flag below — **not** filled with 0, which would fabricate a #1 rank | no | end of day 15 |
| `no_position_data` | 1 if the item never had an impression in the known half | n/a — this *is* the missingness flag | binary indicator | end of day 15 |
| `days_with_impressions` | count of active days, days 1–15 | fill 0 | no | end of day 15 |
| `ctr_pct` | clicks/impressions ×100, days 1–15 | 0 when impressions = 0 (explicit guard, not a silent NaN) | no | end of day 15 |
| `ga4_days_available` | count of days with `ga4_data_available IS TRUE` | fill 0 (genuinely no available days) | no | end of day 15 |
| `pos_top_10` / `pos_11_20` / `pos_21_50` / `pos_50_plus` / `pos_no_data` | one-hot position tier | the `no_data` bucket absorbs missing rather than guessing a tier | yes (one-hot) | end of day 15 |


All eight feature columns are built with `report_date <= 2026-03-15`, nothing in this table can see days 16-31, where the table lives

In [3]:
# Framing table above is verified structurally: confirm no feature column
# depends on outcome-half data by checking the SQL WHERE clause was applied (already shown
# in Section 1's query) and that FEATURES excludes outcome_impressions entirely.
assert "outcome_impressions" not in FEATURES
print("Confirmed: outcome_impressions is not in the feature list.")


Confirmed: outcome_impressions is not in the feature list.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Date Ranges:** all features above are derived using `report_date <= 2026-03-15` (known half). The label (`declined`) is derived using `report_date > 2026-03-15` (outcome half). No overlap.

**Product Flags:** in this data cut, only the observations of GSC/GA4 metrics are provided. No calculated health score/decision flags/"needs review" columns are provided at this grain level, so there's nothing to inadvertently train on in this set.

**Label Derived Columns:** the only possibility is `outcome_impressions` (the metric the label is derived based on) - already excluded from `FEATURES` above. I test this below by including this column deliberately in addition to what's shown below, similar to the leakage example in the contract notebook, but continue to additional tests not covered in that notebook - base rate, feature importance, and random vs. grouped split.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score

X = df[FEATURES]
y = df["declined"]
groups = df["client_hash_id"]

base_rate = y.mean()
print(f"Base rate (declined=1): {base_rate*100:.1f}%  <- naive 'always predict majority' baseline")

# --- Test 1: random split (honest features only) ---
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)
rand_auc = roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])
rand_acc = accuracy_score(y_te, rf.predict(X_te))
print(f"\nRANDOM split   accuracy={rand_acc:.3f}  auc={rand_auc:.3f}  (base rate {base_rate:.3f})")

# --- Test 2: grouped split by client (the honest question: does it work on a client it never saw?) ---
gkf = GroupKFold(n_splits=5)
group_aucs = []
for tr_idx, te_idx in gkf.split(X, y, groups=groups):
    m = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    group_aucs.append(roc_auc_score(y.iloc[te_idx], m.predict_proba(X.iloc[te_idx])[:, 1]))
group_auc = np.mean(group_aucs)
print(f"GROUPED split  auc={group_auc:.3f}  (avg over 5 client-holdout folds)")
print(f"Gap (random - grouped): {rand_auc - group_auc:+.3f}")


Base rate (declined=1): 61.0%  <- naive 'always predict majority' baseline

RANDOM split   accuracy=0.777  auc=0.817  (base rate 0.610)
GROUPED split  auc=0.771  (avg over 5 client-holdout folds)
Gap (random - grouped): +0.046


In [5]:
# --- Test 3: feature importance sanity check ---
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importances (honest model):")
print(importances.to_string())
print("\nNo single feature should tower over the rest -- if one does, that's the signal to")
print("go re-check where it comes from, the same way outcome_impressions gets caught below.")


Feature importances (honest model):
avg_position_filled      0.288291
pos_no_data              0.196762
no_position_data         0.190840
log_impressions          0.149062
days_with_impressions    0.103501
pos_top_10               0.030195
ctr_pct                  0.019366
clicks                   0.013213
ga4_days_available       0.004319
pos_21_50                0.003619
pos_11_20                0.000478
pos_50_plus              0.000354

No single feature should tower over the rest -- if one does, that's the signal to
go re-check where it comes from, the same way outcome_impressions gets caught below.


In [6]:
# --- Test 4: deliberately add the label-derived column back and watch the score jump ---
FEATURES_LEAKY = FEATURES + ["outcome_impressions"]
Xl = df[FEATURES_LEAKY]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, y, test_size=0.25, random_state=42, stratify=y)
rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_leaky.fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, rf_leaky.predict_proba(Xl_te)[:, 1])
leaky_acc = accuracy_score(yl_te, rf_leaky.predict(Xl_te))
print(f"WITH outcome_impressions   accuracy={leaky_acc:.3f}  auc={leaky_auc:.3f}")
print(f"WITHOUT it (honest, random split)  accuracy={rand_acc:.3f}  auc={rand_auc:.3f}")
print(f"\nJump in AUC from adding the label-derived column: {leaky_auc - rand_auc:+.3f}")

leaky_importances = pd.Series(rf_leaky.feature_importances_, index=FEATURES_LEAKY).sort_values(ascending=False)
print("\nTop 3 importances with the leak present:")
print(leaky_importances.head(3).to_string())
print("-> outcome_impressions should dominate; that's the confession, not a discovery.")


WITH outcome_impressions   accuracy=0.886  auc=0.976
WITHOUT it (honest, random split)  accuracy=0.777  auc=0.817

Jump in AUC from adding the label-derived column: +0.158

Top 3 importances with the leak present:
outcome_impressions    0.623270
avg_position_filled    0.101246
log_impressions        0.074776
-> outcome_impressions should dominate; that's the confession, not a discovery.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Here’s the list of fields that you ruled out, along with just one reason for each.

outcome_impressions (or anything else calculated only between day 16 to 31) – this is what the label is thresholded on; using it would be circular, proven by the spike in Test 4.

client_hash_id, content_hash_id – identifiers, only used for grouping/merging/splitting; providing an ID would allow the model to memorize certain IDs rather than learn a general rule, something that is caught by the split-grouped test above.

report_date – a raw date would allow the model to rely on “which day of March,” a pattern that does not generalize to any other month or other panel period.

GA4 engagement columns for rows with `ga4_data_available IS TRUE` — this can be done through
  `ga4_days_available` which is a number already, however, there is no raw GA4 metric that does not
  get any filtration, since three-valued nature of the flag would imply generation of phantom data
  from nothing in case of direct reading.
Any precomputed decision flag or health score — there are no precomputed decision flags or health scores
  in this particular database slice, however, were there any, they would act merely as a benchmark to beat.


In [7]:
# Confirm the exclusion list holds against the actual feature set used above.
excluded_check = {"outcome_impressions", "client_hash_id", "content_hash_id", "report_date"}
assert excluded_check.isdisjoint(set(FEATURES)), "An excluded field leaked into FEATURES!"
print("Confirmed: none of the excluded fields are present in the honest FEATURES list.")


Confirmed: none of the excluded fields are present in the honest FEATURES list.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.